# Clustering
---

In [1]:
import pandas as pd

In [2]:
user_features = pd.read_csv("data/data_preprocessed/user_features_grouped1.csv")

In [3]:
df = user_features[user_features["group"] == "Not assigned"]

In [4]:
df.columns

Index(['user_id', 'gender', 'married', 'has_children', 'home_country',
       'home_city', 'home_airport', 'age', 'state', 'num_session', 'num_trips',
       'num_flights', 'num_hotels', 'booking_discount_rate', 'average_seats',
       'average_checked_bags', 'average_seat_price', 'weekend_trip_ratio',
       'average_nights', 'total_flight_cost', 'total_hotels_cost',
       'conversion_rate', 'age_group', 'group', 'total_spent'],
      dtype='str')

In [5]:
df = df.drop(columns=['user_id', 'home_country',
       'home_city', 'home_airport', 'state', 'age_group', "group"])

In [6]:
df["gender"] = df["gender"].map({"F":0, "M":1, "O":2})

In [7]:
df.shape

(3375, 18)

In [14]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Create and fit the scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

In [13]:
df

,gender,married,has_children,age,num_session,num_trips,num_flights,num_hotels,booking_discount_rate,average_seats,average_checked_bags,average_seat_price,weekend_trip_ratio,average_nights,total_flight_cost,total_hotels_cost,conversion_rate,total_spent
0,0,True,False,51,8,2,2,2,0.000000,1.5,0.500000,276.252500,0.500000,1.000000,864.0900,230.0,0.250,1728.180
1,0,True,True,50,8,2,1,2,0.000000,1.0,0.000000,189.910000,1.000000,4.000000,189.9100,2199.0,0.250,379.820
2,0,True,False,43,8,5,5,5,0.200000,1.0,0.400000,248.532000,0.200000,3.800000,1237.6930,2429.0,0.625,2475.386
5,0,False,True,50,8,5,5,5,0.200000,1.2,0.400000,336.316000,0.200000,5.200000,1919.1325,5579.0,0.625,3838.265
6,0,False,True,45,8,3,3,3,0.000000,1.0,0.333333,218.956667,0.000000,1.333333,656.8700,681.0,0.375,1313.740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5988,0,True,False,40,8,3,2,3,0.333333,1.0,0.500000,412.630000,0.333333,4.666667,825.2600,2664.0,0.375,1650.520
5991,0,True,False,44,8,2,2,1,0.500000,1.0,1.500000,316.845000,0.000000,5.000000,633.6900,585.0,0.250,1267.380
5993,0,True,True,49,8,2,2,1,0.500000,1.5,0.000000,448.535000,0.000000,1.000000,1122.1060,220.0,0.250,2244.212
5994,0,True,True,44,8,2,2,2,0.000000,1.0,0.000000,176.675000,1.000000,1.000000,353.3500,291.0,0.250,706.700


In [12]:
X

array([[-0.36106394,  1.1100193 , -0.66506217, ..., -0.97110614,
        -1.09202442, -0.32066768],
       [-0.36106394,  1.1100193 ,  1.50361882, ...,  0.1653985 ,
        -1.09202442, -1.65208399],
       [-0.36106394,  1.1100193 , -0.66506217, ...,  0.29815424,
         1.76635762,  0.41714882],
       ...,
       [-0.36106394,  1.1100193 ,  1.50361882, ..., -0.97687813,
        -1.09202442,  0.18887979],
       [-0.36106394,  1.1100193 ,  1.50361882, ..., -0.935897  ,
        -1.09202442, -1.32931161],
       [-0.36106394, -0.90088523, -0.66506217, ..., -1.02074524,
         0.81356361,  0.02509216]], shape=(3375, 18))

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA


# ============================================================
# 3. FIND OPTIMAL K  (Elbow + Silhouette)
# ============================================================
inertias = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(list(K_range), inertias, marker='o')
ax[0].set_title('Elbow Method')
ax[0].set_xlabel('Number of clusters (k)')
ax[0].set_ylabel('Inertia')
ax[0].grid(True)

ax[1].plot(list(K_range), sil_scores, marker='o', color='green')
ax[1].set_title('Silhouette Score')
ax[1].set_xlabel('Number of clusters (k)')
ax[1].set_ylabel('Score')
ax[1].grid(True)

plt.tight_layout()
plt.savefig('kmeans_k_selection.png', dpi=100)
plt.show()

best_k = list(K_range)[int(np.argmax(sil_scores))]
print(f"\nBest k by silhouette: {best_k}")

# ============================================================
# 4. TRAIN FINAL K-MEANS MODEL
# ============================================================
k = 4  # or use best_k
kmeans = KMeans(
    n_clusters=k,
    init='k-means++',
    n_init=10,
    max_iter=300,
    tol=1e-4,
    random_state=42
)

labels = kmeans.fit_predict(X_scaled)

df['Cluster'] = labels
print("\nCluster counts:")
print(df['Cluster'].value_counts().sort_index())

# Cluster centers (in original scale)
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)
centers_df = pd.DataFrame(centers_original, columns=df.columns[:2])
print("\nCluster centers (original scale):")
print(centers_df)

# ============================================================
# 5. EVALUATE
# ============================================================
sil = silhouette_score(X_scaled, labels)
db = davies_bouldin_score(X_scaled, labels)

print(f"\nInertia (WCSS):          {kmeans.inertia_:.4f}")
print(f"Silhouette Score:        {sil:.4f}   (higher is better, max 1)")
print(f"Davies-Bouldin Index:    {db:.4f}   (lower is better)")
print(f"Iterations to converge:  {kmeans.n_iter_}")

# ============================================================
# 6. VISUALIZE
# ============================================================
plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    df['Feature1'], df['Feature2'],
    c=df['Cluster'], cmap='viridis',
    s=50, alpha=0.7, edgecolor='k'
)

# Plot centroids
plt.scatter(
    centers_original[:, 0], centers_original[:, 1],
    c='red', marker='X', s=250,
    edgecolor='black', linewidth=1.5,
    label='Centroids'
)

plt.title(f'K-Means Clustering (k={k})')
plt.xlabel('Feature1')
plt.ylabel('Feature2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('kmeans_result.png', dpi=100)
plt.show()

# ============================================================
# 7. PREDICT NEW DATA
# ============================================================
new_data = np.array([[0.0, 0.0], [5.0, 5.0], [-5.0, 5.0]])
new_scaled = scaler.transform(new_data)
new_labels = kmeans.predict(new_scaled)

print("\nPredictions for new points:")
for pt, lbl in zip(new_data, new_labels):
    print(f"  {pt} -> Cluster {lbl}")

# ============================================================
# 8. SAVE / LOAD MODEL (optional)
# ============================================================
import joblib

joblib.dump({'kmeans': kmeans, 'scaler': scaler}, 'kmeans_model.pkl')
loaded = joblib.load('kmeans_model.pkl')
print("\nModel saved and reloaded successfully.")

ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values